# 01 · Fuentes y datasets (S1, S4)

**Historias:** US-01.1 (fuentes y licencias), US-01.4 (auditoría), US-01.5 (particiones), US-01.6 (prueba virgen).
**Kernel:** `colibri-yolo`.

Este notebook deja la zona de trabajo (`D:\video_eval\datos`) lista para todos los demás, **partiendo solo del repo**:

| Sección | Qué hace | Sale a | Decisión |
|---|---|---|---|
| 1 | Comprueba los 12 videos (fps, frames, SHA256) y explica cómo conseguirlos | `runs/01/fuentes_verificadas.csv` | D-07 |
| 2 | Descarga ds1 y ds2 de Roboflow con **tu** clave | `datos/roboflow/raw/` | D-01, D-03 |
| 3 | Auditoría: inventario, revisión visual, fuga temporal de ds1 | `runs/01/` | — |
| 4 | Re-partición de ds1 por bloques temporales y materialización | `datos/roboflow/final*/` | D-02 |
| 5 | Prueba vv (S1-S2): 247 frames regenerados desde los videos | `datos/frames/vv_prueba/` | D-04 → D-07 |
| 6 | `humming` v1 y v2: frames regenerados desde los videos + etiquetas del repo, listas de entrenamiento | `datos/frames/humming_*`, `listas_*` | D-07, D-08 |

**Por qué se regeneran los frames:** las imágenes de `humming` son cuadros de videos de Adobe Stock y YouTube, que no
se pueden redistribuir. El repo publica solo las etiquetas, los manifiestos y este código (decisión del 04-nov).
Los frames se decodifican **en orden** (`grab` secuencial), igual que cuando se crearon, para que el índice
`_fNNNNN` caiga exactamente en el mismo cuadro.

In [ ]:
%run 00_comun.ipynb
OUT = RUNS / "01"; OUT.mkdir(parents=True, exist_ok=True)

## 1. Los 12 videos

Colócalos en `datos/fuentes/videos/<carpeta>/<archivo>` con el nombre exacto de `datos_derivados/fuentes.csv`.

- **Adobe Stock** (vv01, vv02, vv03, A160): con licencia, desde la biblioteca del proyecto
  (enlace en la columna `url`). Se descargan a mano.
- **YouTube** (4 `train_` y 4 `test_`): con `yt-dlp` en la resolución indicada en `nota` (720p, 1080p o 2160p).
  La celda de abajo lo hace si pones `DESCARGAR_YOUTUBE = True` y tienes `yt-dlp` en el entorno.

**Lo que debe coincidir** es `fps` y `frames`: de eso dependen los índices de frame de todas las etiquetas. El SHA256
solo coincidirá con el archivo original de Yilber; una descarga nueva de YouTube puede tener otro SHA256 con los mismos
cuadros. Si `frames` no coincide, las etiquetas de ese video **no** son válidas para esa copia.

In [ ]:
DESCARGAR_YOUTUBE = False

def sha256(path, bloque=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(bloque), b""):
            h.update(b)
    return h.hexdigest()

if DESCARGAR_YOUTUBE:
    import subprocess
    for r in FUENTES[FUENTES.origen == "youtube"].itertuples():
        dst = video_path(r.video_id)
        if dst.exists():
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        alto = int(str(r.nota).rstrip("p"))
        subprocess.run([sys.executable, "-m", "yt_dlp", "-f", f"bv*[height={alto}][ext=mp4]+ba[ext=m4a]/b[height={alto}]",
                        "--merge-output-format", "mp4", "-o", str(dst), r.url], check=True)

rows = []
for r in FUENTES.itertuples():
    p = video_path(r.video_id); row = dict(video_id=r.video_id, corto=r.corto, existe=p.exists())
    if p.exists():
        cap = cv2.VideoCapture(str(p))
        row.update(fps=round(cap.get(cv2.CAP_PROP_FPS), 2), frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
                   ancho=int(cap.get(3)), alto=int(cap.get(4)), sha256=sha256(p)); cap.release()
        row["fps_ok"] = abs(row["fps"] - r.fps) < 0.02
        row["frames_ok"] = row["frames"] == r.frames
        row["sha256_ok"] = (row["sha256"] == r.sha256) if isinstance(r.sha256, str) else None
    rows.append(row)
VER = pd.DataFrame(rows); VER.to_csv(OUT / "fuentes_verificadas.csv", index=False)
display(VER.drop(columns="sha256", errors="ignore"))
faltan = VER[~VER.existe].corto.tolist()
print("faltan:", faltan or "ninguno", "| frames distintos:", VER[VER.get("frames_ok", True) == False].corto.tolist())

## 2. Descarga de ds1 y ds2 (US-01.4, CA-01.4.1)

Necesita `ROBOFLOW_API_KEY` en `.env` (copia `.env.example`). Cada persona usa su propia clave; los dos datasets son
públicos. Desde aquí, las secciones 2 a 4 son las del notebook original de S1 (`historico/colibri_yolo26_tracking.ipynb`),
con las rutas de `config.yaml`.

In [ ]:
from collections import defaultdict, Counter
import random, shutil, platform, getpass, datetime as dt
from PIL import Image
# Portado de notebook original colibri_yolo26_tracking.ipynb, celda 3
SEED = 42
random.seed(SEED); np.random.seed(SEED)

WORK = P["roboflow"]
DATA_RAW = WORK / "raw"          # una subcarpeta por dataset
DATA_FINAL = WORK / "final"
REPORTS = OUT
for p in (DATA_RAW, DATA_FINAL, REPORTS):
    p.mkdir(parents=True, exist_ok=True)

# Decisión 2026-11-01: entrenar con dataset 2 (fotos variadas) + dataset 1 (frames de video en vuelo).
# Si haces fork a tu workspace con una versión sin "Stretch", cambia workspace/project/version aquí.
DATASETS = {
    "ds1": dict(workspace="magic-workspace", project="hummingbird-dataset-0zvvn", version=1, format="yolo26",
                tipo="frames de 2 videos", stretched_from_16_9=True),
    "ds2": dict(workspace="modelodevisinartificialcolibres", project="hummingbirds-detect-classify-gcm5s", version=1,
                format="yolo26", tipo="fotos variadas", stretched_from_16_9=False),
}
CLASS_NAMES = {0: "hummingbird"}      # ds1 'hummingbird' y ds2 'colibri' se unifican en la clase 0
COCO_BIRD_ID = 14                     # clase 'bird' en COCO
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
# Auditoría 2026-11-01: la v1 se exporta con "Stretch to 640x640" desde frames 16:9 (1920x1080 y 1280x720).
ORIG_ASPECT = 16 / 9                  # se usa para medir cajas sin deformación y para des-estirar al materializar
UNSTRETCH_SIZE = (640, 360)           # (ancho, alto) al que se devuelven las imágenes; las etiquetas normalizadas no cambian

RUN_RECORD = {"created": dt.datetime.now().isoformat(timespec="seconds"), "seed": SEED,
              "python": platform.python_version(),
              "datasets": {k: dict(v) for k, v in DATASETS.items()}}

In [ ]:
# Portado de notebook original colibri_yolo26_tracking.ipynb, celda 5
import roboflow
from roboflow import Roboflow

api_key = os.environ.get("ROBOFLOW_API_KEY") or getpass.getpass("Roboflow API key: ")
rf = Roboflow(api_key=api_key)
rf_versions, RAW_DIRS = {}, {}
for key, d in DATASETS.items():
    v = rf.workspace(d["workspace"]).project(d["project"]).version(d["version"])
    ds = v.download(d["format"], location=str(DATA_RAW / key), overwrite=False)
    rf_versions[key], RAW_DIRS[key] = v, Path(ds.location)
    RUN_RECORD["datasets"][key].update({"downloaded": dt.datetime.now().isoformat(timespec="seconds"),
                                        "location": ds.location})
del api_key
RUN_RECORD["roboflow_pkg"] = roboflow.__version__
print(RAW_DIRS)

In [ ]:
# Portado de notebook original colibri_yolo26_tracking.ipynb, celda 7
for key, root in RAW_DIRS.items():
    print(f"########## {key}: {DATASETS[key]['project']}")
    for readme in sorted(root.glob("README*.txt")):
        print(f"===== {readme.name} =====")
        print(readme.read_text(encoding="utf-8", errors="replace"))
    for attr in ("license", "preprocessing", "augmentation", "splits", "images"):
        if hasattr(rf_versions[key], attr):
            print(attr, "->", getattr(rf_versions[key], attr))

In [ ]:
# Portado de notebook original colibri_yolo26_tracking.ipynb, celda 8
# La API de Roboflow declara "MIT" (declarada por quien subió el dataset). Los frames salen de dos videos de terceros
# ("videoplayback.mp4" y "190921_SonyClip#1_ParcelaI_visitas.mp4"): la licencia no acredita los derechos de esos videos.
# También se pueden pasar como variables de entorno LICENSE_DS1 / LICENSE_DS2 (útil con run_pipeline.py).
# ds2 declara "CC BY 4.0": exige citar el dataset y su autor en la documentación del proyecto.
LICENSE_CONFIRMED = {"ds1": os.environ.get("LICENSE_DS1"),   # p. ej. "MIT (declarada en Roboflow; procedencia de videos sin verificar)"
                     "ds2": os.environ.get("LICENSE_DS2")}   # p. ej. "CC BY 4.0, atribución en README del proyecto"
assert all(LICENSE_CONFIRMED.values()), "Gate CA-01.4.2: registra la licencia verificada de cada dataset."
for k, lic in LICENSE_CONFIRMED.items():
    RUN_RECORD["datasets"][k]["license"] = lic

## 3. Auditoría (US-01.4)

### 3.1 Inventario: clases, imágenes, instancias, cajas

In [ ]:
# Portado de notebook original colibri_yolo26_tracking.ipynb, celda 10
def load_data_yaml(root: Path) -> dict:
    cfg = yaml.safe_load((root / "data.yaml").read_text())
    names = cfg["names"]
    cfg["names"] = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
    return cfg

def split_dirs(root: Path) -> dict:
    """Devuelve {split: (dir_imagenes, dir_etiquetas)} para la estructura de Roboflow (train/valid/test)."""
    out = {}
    for split in ("train", "valid", "val", "test"):
        img = root / split / "images"
        if img.is_dir():
            out["valid" if split == "val" else split] = (img, root / split / "labels")
    return out

def read_labels(lbl_path: Path) -> list:
    if not lbl_path.exists():
        return []
    rows = []
    for line in lbl_path.read_text().splitlines():
        parts = line.split()
        if len(parts) == 5:
            c, x, y, w, h = parts
            rows.append((int(c), float(x), float(y), float(w), float(h)))
        elif len(parts) > 5:  # polígono: se convierte a caja envolvente
            c, pts = int(parts[0]), np.array(parts[1:], float).reshape(-1, 2)
            (x0, y0), (x1, y1) = pts.min(0), pts.max(0)
            rows.append((c, (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0))
    return rows

def original_stem(img_path: Path) -> str:
    """Roboflow nombra los archivos '<original>_<ext>.rf.<hash>.<ext>': las copias aumentadas comparten el original."""
    return img_path.name.split(".rf.")[0]

import re
FRAME_RE = re.compile(r"^(?P<src>.+?)[-_](?P<frame>\d+)_(?:jpe?g|png)$", re.I)

def video_frame(stem: str):
    """'videoplayback_mp4-0203_jpg' -> ('videoplayback_mp4', 203). Todo el dataset son frames de 2 videos."""
    m = FRAME_RE.match(stem)
    return (m["src"], int(m["frame"])) if m else (stem, -1)

def inventory(root: Path, ds: str):
    img_rows, box_rows = [], []
    for split, (img_dir, lbl_dir) in split_dirs(root).items():
        for img in sorted(p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS):
            with Image.open(img) as im:
                W, H = im.size
            labels = read_labels(lbl_dir / f"{img.stem}.txt")
            stem = original_stem(img)
            src, frame = video_frame(stem) if DATASETS[ds]["stretched_from_16_9"] else (f"{ds}_foto", -1)
            img_rows.append(dict(ds=ds, split=split, path=str(img), stem=stem, src=src, frame=frame, W=W, H=H,
                                 n_boxes=len(labels)))
            for c, x, y, w, h in labels:
                box_rows.append(dict(ds=ds, split=split, path=str(img), cls=c, cx=x, cy=y, w=w, h=h,
                                     area=w * h, aspect=(w * W) / max(h * H, 1e-9), w_px=w * W, h_px=h * H,
                                     aspect_orig=((w * ORIG_ASPECT) / max(h, 1e-9)) if (DATASETS[ds]["stretched_from_16_9"] and W == H)
                                     else (w * W) / max(h * H, 1e-9)))   # aspecto real (ds2 estirado: no recuperable)
    return pd.DataFrame(img_rows), pd.DataFrame(box_rows)

parts = {k: inventory(root, k) for k, root in RAW_DIRS.items()}
for k, root in RAW_DIRS.items():
    names = load_data_yaml(root)["names"]
    print(k, "clases:", names)
    assert len(names) == 1, f"{k}: se esperaba una sola clase; revisa el mapeo a CLASS_NAMES"
imgs = pd.concat([p[0] for p in parts.values()], ignore_index=True)
boxes = pd.concat([p[1] for p in parts.values()], ignore_index=True)
summary = imgs.groupby(["ds", "split"]).agg(imagenes=("path", "size"), sin_etiqueta=("n_boxes", lambda s: int((s == 0).sum())),
                                    multi_ave=("n_boxes", lambda s: int((s > 1).sum())), instancias=("n_boxes", "sum"))
summary["pct"] = (100 * summary["imagenes"] / summary.groupby(level=0)["imagenes"].transform("sum")).round(1)
display(summary)
display(imgs.groupby("ds")[["W", "H"]].describe().T)
display(pd.crosstab(imgs.src, imgs.split, margins=True))   # esperado: 2 videos, ambos en los tres splits
display(boxes.groupby(["ds", "split"])[["w_px", "h_px", "area", "aspect", "aspect_orig"]].describe().T.round(3))

In [ ]:
# Portado de notebook original colibri_yolo26_tracking.ipynb, celda 11
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(np.sqrt(boxes["area"]), bins=40); ax[0].set_title("sqrt(área normalizada) de cajas")
ax[1].hist(boxes["aspect_orig"], bins=40); ax[1].set_title("aspecto w/h en el frame 16:9 original")
ax[2].scatter(boxes["cx"], boxes["cy"], s=3, alpha=.4); ax[2].invert_yaxis(); ax[2].set_title("centros de caja")
plt.tight_layout(); plt.savefig(REPORTS / "dataset_cajas.png", dpi=120); plt.show()
"""Ojo: boxes["aspect"] está medido sobre la imagen estirada a 640x640 y sale 16/9 veces más estrecho; usar aspect_orig.
Lectura sugerida (CA-01.4.5): una distribución de aspecto bimodal suele separar aves posadas
(cajas más altas) de aves en vuelo con alas extendidas (cajas más anchas). Esa variación de caja
es la que desplazará el centro en el video y debe anotarse como limitación de la referencia corporal."""

### 3.2 Revisión visual con semilla fija (CA-01.4.4)

In [ ]:
# Portado de notebook original colibri_yolo26_tracking.ipynb, celda 13
def show_grid(paths, lbl_lookup, n_cols=6, title="", out=None):
    n = len(paths); rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(rows, n_cols, figsize=(3 * n_cols, 3 * rows))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, p in zip(np.atleast_1d(axes).ravel(), paths):
        im = Image.open(p).convert("RGB"); W, H = im.size
        ax.imshow(im)
        for c, x, y, w, h in lbl_lookup(Path(p)):
            ax.add_patch(plt.Rectangle(((x - w / 2) * W, (y - h / 2) * H), w * W, h * H, fill=False, lw=2, ec="lime"))
            ax.plot(x * W, y * H, "r+", ms=10)          # centro de caja = referencia corporal (US-01.2)
        ax.set_title(Path(p).name[:22], fontsize=7)
    fig.suptitle(title); plt.tight_layout()
    if out: plt.savefig(out, dpi=110)
    plt.show()

def raw_label_lookup(img_path: Path):
    return read_labels(img_path.parent.parent / "labels" / f"{img_path.stem}.txt")

for split in imgs["split"].unique():
    sample = imgs[imgs.split == split].sample(min(24, (imgs.split == split).sum()), random_state=SEED)["path"].tolist()
    show_grid(sample, raw_label_lookup, title=f"Muestra {split}", out=REPORTS / f"muestra_{split}.png")

REVIEW_NOTES = {  # completa tras mirar las rejillas
    "todas_las_aves_etiquetadas": None,
    "criterio_caja_consistente": None,
    "cajas_incluyen_alas_extendidas": None,
    "observaciones": "",
}

### 3.3 Fugas y casi-duplicados (CA-01.4.6)

In [ ]:
# Portado de notebook original colibri_yolo26_tracking.ipynb, celda 15
# Auditoría 2026-11-01: el dataset son frames consecutivos (paso de 1 frame) de 2 videos con cámara fija,
# repartidos al azar entre splits. Cada imagen de valid/test está a 1-4 frames de una de train: fuga casi total.
# Un pHash con unión transitiva encadenaría cada video entero en un solo grupo, así que la fuga se mide
# por vecindad temporal dentro del mismo video.
LEAK_GAP_FRAMES = 15      # dos frames a <= 15 frames (~0.5 s) se consideran casi-duplicados

def min_gap_to(df: pd.DataFrame, split_col: str, target: str = "train") -> pd.Series:
    """Distancia en frames de cada imagen a la imagen más cercana del mismo video asignada a `target`."""
    out = pd.Series(np.inf, index=df.index)
    for src, g in df[df.frame >= 0].groupby("src"):
        ref = np.sort(g.loc[g[split_col] == target, "frame"].values)
        if len(ref) == 0:
            continue
        pos = np.searchsorted(ref, g.frame.values)
        lo = ref[np.clip(pos - 1, 0, len(ref) - 1)]; hi = ref[np.clip(pos, 0, len(ref) - 1)]
        out[g.index] = np.minimum(np.abs(g.frame.values - lo), np.abs(g.frame.values - hi))
    return out

imgs = imgs.reset_index(drop=True)
imgs["gap_to_train"] = min_gap_to(imgs, "split")
# Solo ds1 son frames de video; en ds2 (fotos) la auditoría por embeddings dio 0,8 % de casi-duplicados entre splits.
ev = imgs[(imgs.ds == "ds1") & imgs.split.isin(["valid", "test"])]
leaks = ev[ev.gap_to_train <= LEAK_GAP_FRAMES]
leaky_groups = leaks.index          # se mantiene el nombre que usa la sección 4
print(f"ds1: valid/test a <= {LEAK_GAP_FRAMES} frames de train: {len(leaks)} de {len(ev)} ({len(leaks)/max(len(ev),1):.0%})")
display(ev.groupby("split").gap_to_train.describe())
# Diagnóstico visual: un frame de test junto a sus vecinos de train
if len(leaks):
    r = leaks.iloc[0]
    neigh = imgs[(imgs.src == r.src) & (imgs.split == "train")].assign(d=lambda d: (d.frame - r.frame).abs()).nsmallest(3, "d")
    show_grid([r.path] + neigh.path.tolist(), raw_label_lookup, n_cols=4, title=f"Fuga: {r.split} frame {r.frame} y vecinos de train")

## 4. Particiones (US-01.5)

**ds2** (fotos, sin fugas) conserva su distribución de Roboflow. Lo que sigue aplica a **ds1**.

Regla documentada (CA-01.5.1): se usa la distribución de Roboflow **solo si** no hay fugas y valid/test tienen al menos `MIN_EVAL_IMAGES`.
La auditoría del 2026-11-01 encontró fuga casi total (frames vecinos del mismo video en train y en valid/test), así que se **re-particiona por bloques temporales**:
cada video se corta en tramos contiguos de unos `CHUNK_IMAGES` frames, los tramos se reparten 70/15/15 **dentro de cada video** (los dos videos aparecen en los tres splits)
y se descartan de valid/test los frames a menos de `GUARD_FRAMES` de un tramo de train.

Limitación que no se puede quitar: ambos videos son de cámara fija, así que valid/test comparten fondo con train. La prueba mide
"mismo escenario, otro momento", no generalización a una escena nueva; para eso está el reporte cruzado entre videos y, después, el video del proyecto.

In [ ]:
# Portado de notebook original colibri_yolo26_tracking.ipynb, celda 17
MIN_EVAL_IMAGES = 60
RESPLIT_RATIOS = {"train": 0.70, "valid": 0.15, "test": 0.15}
SEGMENT_GAP = 15       # un salto > 15 frames abre un tramo nuevo
CHUNK_IMAGES = 40      # tamaño aproximado de bloque dentro de un tramo largo
GUARD_FRAMES = 15      # banda de guarda entre bloques de distinto split

def temporal_blocks(df: pd.DataFrame) -> pd.Series:
    """Id de bloque temporal por imagen: tramos contiguos de cada video, troceados en ~CHUNK_IMAGES."""
    out = pd.Series(index=df.index, dtype=object)
    for src, g in df.sort_values(["src", "frame"]).groupby("src"):
        seg = (g.frame.diff().fillna(0) > SEGMENT_GAP).cumsum()
        k = 0
        for _, gg in g.groupby(seg):
            for part in np.array_split(gg.index.values, max(1, round(len(gg) / CHUNK_IMAGES))):
                out[part] = f"{src}#{k}"; k += 1
    return out

def block_split(df: pd.DataFrame, ratios: dict, seed: int, tries: int = 3000) -> pd.Series:
    """Asigna bloques a splits por video; elige, entre `tries` sorteos con semilla fija, el más cercano a las cuotas."""
    rng, out = np.random.default_rng(seed), pd.Series(index=df.index, dtype=object)
    for src, g in df.groupby("src"):
        sizes = g.groupby("block").size(); best = None
        for _ in range(tries):
            lab = rng.choice(list(ratios), size=len(sizes), p=list(ratios.values()))
            n = {k: sizes[lab == k].sum() for k in ratios}
            if min(n.values()) == 0:
                continue
            err = sum(abs(n[k] / len(g) - ratios[k]) for k in ratios)
            if best is None or err < best[0]:
                best = (err, dict(zip(sizes.index, lab)))
        out[g.index] = g.block.map(best[1])
    return out

v1 = imgs.ds == "ds1"
sizes = imgs[v1].split.value_counts()
use_original = len(leaky_groups) == 0 and all(sizes.get(s, 0) >= MIN_EVAL_IMAGES for s in ("valid", "test"))
imgs["block"] = None; imgs["final_split"] = imgs["split"]          # ds2: distribución original
imgs.loc[v1, "block"] = temporal_blocks(imgs[v1])
if not use_original:
    imgs.loc[v1, "final_split"] = block_split(imgs[v1], RESPLIT_RATIOS, SEED)

# Banda de guarda: fuera de valid/test lo que quede a <= GUARD_FRAMES de train
imgs["gap_final"] = min_gap_to(imgs, "final_split")
guard = v1 & imgs.final_split.isin(["valid", "test"]) & (imgs.gap_final <= GUARD_FRAMES)
imgs.loc[guard, "final_split"] = "dropped_guard"

# CA-01.5.2: en valid/test solo una imagen por original de Roboflow. Ninguna versión trae aumentos; solo se aplica
# a ds1, porque en ds2 los nombres originales se repiten entre fotos distintas (4675 únicos de 6766).
eval_mask = v1 & imgs.final_split.isin(["valid", "test"])
dupe_aug = eval_mask & imgs.duplicated(subset=["stem"], keep="first")
imgs.loc[dupe_aug, "final_split"] = "dropped_aug_eval"

SPLIT_DECISION = {"use_original_split": bool(use_original), "leaky_eval_images": int(len(leaky_groups)),
                  "method": None if use_original else "bloques temporales por video + banda de guarda",
                  "segment_gap": SEGMENT_GAP, "chunk_images": CHUNK_IMAGES, "guard_frames": GUARD_FRAMES,
                  "min_eval_images": MIN_EVAL_IMAGES, "ratios": None if use_original else RESPLIT_RATIOS,
                  "dropped_guard": int(guard.sum()), "dropped_augmented_from_eval": int(dupe_aug.sum())}
print(SPLIT_DECISION)
display(pd.crosstab([imgs.ds, imgs.src, imgs.split], imgs.final_split, margins=True))
assert (imgs.loc[v1 & imgs.final_split.isin(["valid", "test"]), "gap_final"] > GUARD_FRAMES).all()
for s in ("valid", "test"):
    if (v1 & (imgs.final_split == s)).sum() < MIN_EVAL_IMAGES:
        print(f"Aviso: {s} tiene menos de {MIN_EVAL_IMAGES} imágenes; ajusta CHUNK_IMAGES o GUARD_FRAMES y documenta.")

### 4.1 Materialización

Escribe `final/` (ds2 + ds1 re-partido) y `final_ds2/` (ablación). Cada uno trae `manifest.csv` con el SHA1 de cada
imagen; el SHA1 del manifiesto completo identifica el dataset y se compara con el de S1.

In [ ]:
# Portado de notebook original colibri_yolo26_tracking.ipynb, celda 18
def materialize(df: pd.DataFrame, dst: Path, names: dict) -> Path:
    if dst.exists():
        shutil.rmtree(dst)
    manifest = []
    for _, r in df[df.final_split.isin(["train", "valid", "test"])].iterrows():
        src = Path(r.path); lbl = src.parent.parent / "labels" / f"{src.stem}.txt"
        out_name = f"{r.ds}__{src.name}"                       # prefijo: evita choques de nombre entre datasets
        (dst / r.final_split / "images").mkdir(parents=True, exist_ok=True)
        (dst / r.final_split / "labels").mkdir(parents=True, exist_ok=True)
        out_img = dst / r.final_split / "images" / out_name
        with Image.open(src) as im:
            if UNSTRETCH_SIZE and DATASETS[r.ds]["stretched_from_16_9"] and im.size[0] == im.size[1]:
                # deshace el "Stretch to 640x640": mismo aspecto 16:9 que el video, que YOLO letterboxea en inferencia
                im.convert("RGB").resize(UNSTRETCH_SIZE, Image.BICUBIC).save(out_img, quality=95)
            else:
                shutil.copy2(src, out_img)
        rows = read_labels(lbl)                                  # toda clase -> 0 (CLASS_NAMES)
        (dst / r.final_split / "labels" / f"{Path(out_name).stem}.txt").write_text(
            "\n".join(f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for _, x, y, w, h in rows))
        manifest.append({"ds": r.ds, "split": r.final_split, "file": out_name, "sha1": hashlib.sha1(out_img.read_bytes()).hexdigest(),
                         "src": r.src, "frame": int(r.frame), "block": r.block})
    data_yaml = {"path": str(dst.resolve()), "train": "train/images", "val": "valid/images", "test": "test/images",
                 "names": names}
    (dst / "data.yaml").write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True))
    man = pd.DataFrame(manifest).sort_values(["split", "file"])
    man.to_csv(dst / "manifest.csv", index=False)
    digest = hashlib.sha1(man.to_csv(index=False).encode()).hexdigest()
    return dst / "data.yaml", digest

DATA_YAML, DATASET_DIGEST = materialize(imgs, DATA_FINAL, CLASS_NAMES)
# Ablación (ds2 solo) para medir si los frames de video de ds1 aportan en la prueba propia
DATA_YAML_DS2, DIGEST_DS2 = materialize(imgs[imgs.ds == "ds2"], WORK / "final_ds2", CLASS_NAMES)
RUN_RECORD.setdefault("splits", {}).update({"split_decision": SPLIT_DECISION, "final_yaml": str(DATA_YAML),
                              "manifest_sha1": DATASET_DIGEST, "manifest_sha1_ds2": DIGEST_DS2, "review_notes": REVIEW_NOTES})
print(DATA_YAML, DATASET_DIGEST)

In [ ]:
ref_rr = P["resultados"] / "S1_datos" / "run_record.json"
if ref_rr.exists():
    rr = json.loads(ref_rr.read_bytes().decode("utf-8", errors="replace"))
    s1 = rr.get("splits", {})
    for k, mine in (("manifest_sha1", DATASET_DIGEST), ("manifest_sha1_ds2", DIGEST_DS2)):
        print(("✓" if s1.get(k) == mine else "✗"), k, "S1:", s1.get(k), "| ahora:", mine)
(OUT / "run_record.json").write_text(json.dumps(RUN_RECORD, indent=2, default=str, ensure_ascii=False), encoding="utf-8")

## 5. Prueba vv (S1-S2)

247 frames de vv01-vv03 (cada 8 o 5 frames, `datos_derivados/vv/manifest_frames.csv`) con las etiquetas corregidas
en Roboflow (`prueba_etiquetas_corregidas.zip`): 174 cajas y 73 frames vacíos. Se excluyen 4 frames de vv02
(`ignorar_frames.txt`) donde solo asoma el pico o el ala. Fue la prueba de decisión de S2 (D-05) hasta que D-07 pasó
vv a entrenamiento. Sustituye a `build_video_test`, que necesitaba los zips de frames.

In [ ]:
# Portado de video-val/reentreno/preparar_frames_roboflow.py
def iter_frames(path, step, wanted=None):
    """Decodifica en orden; devuelve (idx, t_s, frame) cada `step` frames o solo los índices de `wanted`."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir {path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    idx = 0
    while True:
        ok = cap.grab()
        if not ok:
            break
        if (wanted is None and idx % step == 0) or (wanted is not None and idx in wanted):
            ok, frame = cap.retrieve()
            if ok:
                yield idx, idx / fps, frame
        idx += 1
    cap.release()

def regenerar(video_id, indices, nombre, destino, max_side=None):
    """Escribe <destino>/<nombre(i)>.jpg para cada índice, decodificando el video en orden. Devuelve los escritos."""
    destino.mkdir(parents=True, exist_ok=True)
    falta = {i for i in indices if not (destino / f"{nombre(i)}.jpg").exists()}
    for idx, _, fr in iter_frames(video_path(video_id), 1, falta) if falta else ():
        if max_side:
            h, w = fr.shape[:2]; s = max_side / max(h, w)
            if s < 1:
                fr = cv2.resize(fr, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(destino / f"{nombre(idx)}.jpg"), fr, [cv2.IMWRITE_JPEG_QUALITY, 95])
    return [i for i in indices if (destino / f"{nombre(i)}.jpg").exists()]

import zipfile
TEST_VIDEO = P["frames"] / "vv_prueba"
with zipfile.ZipFile(VV / "prueba_etiquetas_corregidas.zip") as z:
    IGNORAR_VV = set(z.read("ignorar_frames.txt").decode().split())
    ETQ_VV = {Path(n).stem: z.read(n).decode() for n in z.namelist() if n.startswith("labels/") and n.endswith(".txt")}
MF = pd.read_csv(VV / "manifest_frames.csv")
MF["stem"] = MF.file.str.replace(".jpg", "", regex=False)
MF = MF[~MF.stem.isin(IGNORAR_VV)]
(TEST_VIDEO / "labels").mkdir(parents=True, exist_ok=True)
for vid, g in MF.groupby("video_id"):
    ok = regenerar(vid, g.frame.tolist(), lambda i, vid=vid: f"{vid}_f{i:05d}", TEST_VIDEO / "images")
    print(vid, len(ok), "de", len(g), "frames")
for s in MF.stem:
    (TEST_VIDEO / "labels" / f"{s}.txt").write_text(ETQ_VV[s])
print("prueba vv:", len(MF), "frames,", sum(len(ETQ_VV[s].split()) // 5 for s in MF.stem), "cajas (esperado 243 frames y 174 cajas menos las de los ignorados)")

## 6. `humming` v1 y v2 (S4, S5)

**Selección de frames (S4):** `historico/scripts/preparar_frames_roboflow.py` eligió los frames de los `train_` y
`test_` con esta política (parámetros en `datos_derivados/humming/resumen_seleccion.txt`):

- candidatos cada 0,1 s; se guarda uno si pasaron ≥ 0,4 s (0,2 s en frames difíciles para el detector) y la escena
  cambió ≥ 0,3 % de píxeles, o si pasaron ≥ 1,5 s; tope 120 por video;
- `test_`: 3 fps fijos; valid: último 20 % de cada `train_`, con 1 s de guarda;
- fuga: dHash de cada frame de prueba frente a todos los de entrenamiento, distancia ≤ 6 → **0 alertas**.

El resultado está congelado en `manifest_reentrenamiento.csv` y `seleccion_vv.csv`. Las cajas se corrigieron a mano
en Roboflow (`yilber-lopez/humming`, privado); las versiones 1 y 2 están congeladas en
`datos_derivados/humming/etiquetas_{v1,v2}/`. Los cambios de v1 a v2 están en `cambios_v1_v2.csv`.

**Re-partición (`reparticion.py`):** el split sale de los manifiestos, no de Roboflow; las imágenes se reducen a un
lado máximo de 1280 px (las etiquetas normalizadas no cambian); los 4 frames ignorados de vv02 no entran en
train/valid; humming se repite ×3 en `train_ft.txt` junto a ds2+ds1.

In [ ]:
# Portado de video_eval/reentreno/v2/reparticion.py
MAX_SIDE = 1280
OVERSAMPLE = 3

def materializar_humming(version):
    m = cargar_manifiesto(version)
    for (vid, split), g in m.groupby(["video_id", "split"]):
        d = P["frames"] / f"humming_{version}" / split
        regenerar(vid, g.frame.tolist(), lambda i, vid=vid: f"{vid}_f{i:05d}", d / "images", MAX_SIDE)
        (d / "labels").mkdir(parents=True, exist_ok=True)
        for r in g.itertuples():
            shutil.copyfile(r.etiqueta, d / "labels" / r.etiqueta.name)
    m["existe"] = [p.exists() for p in m.imagen]
    print(f"humming {version}: {int(m.existe.sum())} de {len(m)} imágenes;", m.groupby(["split", "origen"]).size().to_dict())
    return m

def listas(version, m):
    """Final de main() de reparticion.py: train_ft (ds2+ds1 + humming x3), val_ft, test_video y los yaml."""
    L = P["frames"] / f"listas_{version}"; L.mkdir(parents=True, exist_ok=True)
    listed = lambda split: [str(p) for p in m[m.split == split].imagen]
    old_train = sorted(str(p) for p in (DATA_FINAL / "train" / "images").glob("*.jpg"))
    old_val = sorted(str(p) for p in (DATA_FINAL / "valid" / "images").glob("*.jpg"))
    (L / "train_ft.txt").write_text("\n".join(old_train + listed("train") * OVERSAMPLE) + "\n")
    (L / "val_ft.txt").write_text("\n".join(old_val + listed("valid")) + "\n")
    (L / "test_video.txt").write_text("\n".join(listed("test")) + "\n")
    (L / "data_ft.yaml").write_text(f"train: {L / 'train_ft.txt'}\nval: {L / 'val_ft.txt'}\ntest: {L / 'test_video.txt'}\nnames:\n  0: hummingbird\n")
    (L / "data_test_video.yaml").write_text(f"train: {L / 'test_video.txt'}\nval: {L / 'test_video.txt'}\ntest: {L / 'test_video.txt'}\nnames:\n  0: hummingbird\n")
    (L / "data_ds2_test.yaml").write_text(f"path: {WORK / 'final_ds2'}\ntrain: test/images\nval: test/images\ntest: test/images\nnames:\n  0: hummingbird\n")
    print(f"listas {version}: train_ft {len(old_train)} ds2+ds1 + {len(listed('train'))} humming x{OVERSAMPLE}; "
          f"val_ft {len(old_val)} + {len(listed('valid'))}; test_video {len(listed('test'))}")

HUM = {}
for version in ("v1", "v2"):
    HUM[version] = materializar_humming(version)
    listas(version, HUM[version])

### 6.1 ¿Los frames regenerados son los mismos?

Si en este equipo siguen las imágenes originales de S4 (`D:\video_eval\reentreno\v2\humming`), se compara una
muestra: diferencia media por píxel (0-255) y distancia dHash. Esperado: diferencia pequeña (recompresión JPEG) y
dHash ≤ 2. Un dHash alto indica que el índice de frame no cae en el mismo cuadro, y entonces las etiquetas de ese
video no valen para esa copia.

In [ ]:
ORIGINALES = {"v1": ZONA / "reentreno" / "v2" / "humming", "v2": ZONA / "reentreno" / "v2" / "eval_gt2" / "humming"}

def dhash(fr):
    g = cv2.cvtColor(cv2.resize(fr, (9, 8), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY)
    return int("".join("1" if x else "0" for x in (g[:, 1:] > g[:, :-1]).flatten()), 2)

for version, raiz in ORIGINALES.items():
    if not raiz.exists():
        print(f"{version}: no hay originales en {raiz}; se omite"); continue
    m = HUM[version].groupby("video_id", group_keys=False).apply(lambda g: g.sample(min(3, len(g)), random_state=SEED))
    rows = []
    for r in m.itertuples():
        o = cv2.imread(str(raiz / r.split / "images" / r.imagen.name)); n = cv2.imread(str(r.imagen))
        if o is None or n is None or o.shape != n.shape:
            rows.append(dict(video=corto(r.video_id), frame=r.frame, ok=False)); continue
        rows.append(dict(video=corto(r.video_id), frame=r.frame, dif_media=float(cv2.absdiff(o, n).mean()),
                         dhash=bin(dhash(o) ^ dhash(n)).count("1")))
    V = pd.DataFrame(rows); V["ok"] = V.get("dhash", pd.Series(dtype=float)).le(2)
    print(f"{version}: {int(V.ok.sum())} de {len(V)} frames coinciden"); display(V.groupby("video")[["dif_media", "dhash"]].max().round(2))

## Conclusiones

- **S1 (D-01..D-03):** ds1 tenía fuga casi total (frames vecinos en train y en valid/test), así que se re-partió por
  bloques temporales de 40 frames con 15 de guarda (442/83/82). ds2 conserva su reparto. Se entrena ds2+ds1.
- **S4 (D-07):** vv y los `train_` pasan a entrenamiento; los 4 `test_` son la prueba y no entran nunca en train ni
  en valid. 0 alertas de fuga por dHash.
- **Replicabilidad:** con los 12 videos y una clave de Roboflow, este notebook reconstruye todos los datos de entrada;
  lo único que no se puede regenerar son los juicios humanos (etiquetas y revisión a ciegas), que están congelados.
- **Pendiente (US-01.1):** confirmar la licencia de ds1 (CA-01.1.5) y la lista blanca de videos para el logo.